In [1]:
import pandas as pd
import numpy as np

In [2]:
# initial data processing
#   cascading the sentence number down to the sentences below it
#   necessary for dividing into training and testing sets
raw_dataset = pd.read_csv('ner_dataset2.csv', na_filter=False, dtype=str)
raw_dataset['Sentence Start'] = ~(raw_dataset['Sentence #'] == '')
raw_dataset['Sentence #'] = raw_dataset['Sentence #'].str.extract(r'(\d+)', expand=False).ffill().astype('int64')

In [3]:
def fast_generate_ngram_model(training_set, gram_size=4):
    modified = pd.DataFrame(training_set)

    for n in range(gram_size - 1):
        modified[f'Previous POS {n + 1}'] = modified['POS'].shift(n + 1)
        modified[f'Previous POS {n + 1}'] = modified[f'Previous POS {n + 1}'].where(modified['Sentence #'] == modified['Sentence #'].shift(n + 1))

    ngram_map = {}

    def write_to_map(row):
        pattern = (*list(map(lambda n : row[f'Previous POS {n}'], list(range(1, gram_size))[::-1])), row['Word'])
        if pattern in ngram_map:
            ngram_map[pattern] += 1
        else:
            ngram_map[pattern] = 1

    modified.apply(write_to_map, axis=1)

    return ngram_map

In [4]:
testing_set = raw_dataset.loc[raw_dataset['Sentence #'].between(1, 1000)]
training_set = raw_dataset.loc[~raw_dataset['Sentence #'].between(1, 1000)]

In [5]:
map = fast_generate_ngram_model(training_set)